# 05 · Model optimisation and final model selection

**SkillPath** (IT3051 Fundamentals of Data Mining, group KND_12)

Stage 7. Four experiment phases, then the final model is chosen, fitted on the whole
training split and evaluated **once** on the held-out respondents.

| Phase | Question | Script |
|---|---|---|
| A | Does handling the 155:1 imbalance help, and how? | `tune_models.py --only A` |
| B | Do the Stage 4 preprocessing and feature choices earn their place? | `--only B` |
| C | What do the hyper-parameters buy? | `--only C` |
| D | Tuned vs baseline — which model ships? | `--only D` |
| — | Fit it, open the test split once | `finalise_model.py` |

```
python scripts/tune_models.py        # phases A-D, appends to reports/model_experiments.csv
python scripts/finalise_model.py     # final fit + the one test evaluation
```

Phases A–C never touch `test.parquet`. Only `finalise_model.py` opens it, and only after
the model has been chosen.

In [1]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from skillpath import config as C
from skillpath import modelling as M
from skillpath import viz

viz.setup()
pd.set_option("display.width", 220, "display.max_columns", 60)


def read(name):
    p = C.REPORTS_DIR / name
    return pd.read_csv(p) if p.exists() else None


stage6 = read("stage6_model_comparison.csv")
phase_a = read("stage7_a_imbalance.csv")
phase_b = read("stage7_b_features.csv")
phase_c = read("stage7_c_tuned.csv")
phase_d = read("stage7_d_tuned_vs_baseline.csv")
best_params = json.loads((C.REPORTS_DIR / "stage7_best_params.json").read_text())
decision = json.loads((C.REPORTS_DIR / "stage7_final_selection.json").read_text())

## Phase A · Imbalance handling

Stage 6 showed per-class F1 tracking class size almost monotonically, so this is the
first thing to attack. Three treatments, tested on two different algorithms so the answer
is not an artefact of one of them:

1. **Nothing** — the untouched 155:1 distribution.
2. **`class_weight`** — re-weight the loss so an error on a rare role costs as much as an
   error on Full-stack. Free: no extra rows.
3. **Random oversampling to 10%** — duplicate real respondents until every role reaches at
   least 10% of Full-stack's size.

**SMOTE was considered and rejected.** It builds synthetic rows by interpolating between
neighbours; 403 of the 479 features are 0/1 skill flags, so an interpolated row says a
respondent "0.4 knows React", which is not a skill profile anybody could have. Duplicating
real respondents keeps every training row valid.

**Full balancing was rejected on cost**, not on principle: it copies the 47 UX/UI
respondents 155 times each and grows the training set to 145,580 rows, over an hour per
fit here.

The resampling lives *inside* the estimator (`modelling.OversampledClassifier`), so it
runs on each fold's training part only. Oversampling before the split is a classic leak —
copies of the same respondent land in both training and validation, and the score rises
for no real reason.

In [2]:
phase_a[["model", "f1_macro", "f1_macro_std", "balanced_accuracy", "accuracy",
         "top3_accuracy", "family_f1_macro", "fit_seconds"]].round(4)

,model,f1_macro,f1_macro_std,balanced_accuracy,accuracy,top3_accuracy,family_f1_macro,fit_seconds
0,Logistic Regression + no imbalance handling,0.3003,0.0081,0.2865,0.5480,0.8297,0.4196,106.2616
1,Logistic Regression + oversampling to 10%,0.2914,0.0041,0.3073,0.5123,0.8089,0.4050,16.8593
2,Random Forest + class_weight,0.2548,0.0070,0.2414,0.5535,0.8294,0.3733,3.2074
3,Logistic Regression + class_weight,0.2535,0.0017,0.3288,0.3650,0.7269,0.3484,73.4602
4,Random Forest + oversampling to 10%,0.2153,0.0042,0.1920,0.5410,0.8263,0.3452,3.3256
5,Random Forest + no imbalance handling,0.1869,0.0063,0.1738,0.5411,0.8201,0.3218,2.6819


## Phase B · Feature set and feature selection

Each row changes exactly one Stage 4 decision and keeps everything else fixed, run on
Logistic Regression because a linear model reacts most visibly to the feature set.

* **want lists removed** — are "technologies I want to learn" worth their 223 columns, or
  are they just a noisy copy of what people already use? (EDA found phi 0.18–0.76 between
  the pairs, so they are related but not duplicates.)
* **rare pooling at 0.25% / 1%** — the Stage 4 threshold was 0.5%.
* **SelectKBest chi² k ∈ {100, 200, 300, all}** — selection sits inside the pipeline, so
  the k best features are chosen from each fold's training part. Selecting once on the
  full training set is the single most common way to leak in a project like this: the
  features are chosen using labels the validation fold is meant to be testing.
* **core+context** — adds OrgSize, Industry, RemoteWork and ICorPM. Stage 4 kept these out
  by default because students, the main users, have no current employer to describe.

In [3]:
phase_b[["model", "note", "f1_macro", "f1_macro_std", "top3_accuracy",
         "family_f1_macro", "fit_seconds"]].round(4)

,model,note,f1_macro,f1_macro_std,top3_accuracy,family_f1_macro,fit_seconds
0,core+context (506 features),phase B: adds OrgSize/Industry/RemoteWork/ICorPM,0.2605,0.0049,0.7413,0.3614,3.4389
1,rare pooling 1%,phase B: fewer technology columns,0.2547,0.0025,0.7299,0.3546,3.7543
2,core (reference),"phase B: 479 features, the Stage 4 default",0.2541,0.0019,0.7301,0.3543,3.6881
3,rare pooling 0.25%,phase B: more technology columns,0.2528,0.0019,0.7296,0.3542,2.5355
4,want lists removed,phase B: does 'what they want to learn' add an...,0.2451,0.0015,0.7008,0.3402,1.5681
5,SelectKBest chi2 k=300,phase B: selection fitted inside every fold,0.2353,0.0029,0.7102,0.3282,1.8359
6,SelectKBest chi2 k=200,phase B: selection fitted inside every fold,0.2315,0.0039,0.6868,0.3186,1.5942
7,SelectKBest chi2 k=all,phase B: selection fitted inside every fold,0.2290,0.0040,0.7353,0.3301,2.0883
8,SelectKBest chi2 k=100,phase B: selection fitted inside every fold,0.2277,0.0043,0.6489,0.3072,1.3768


In [4]:
if phase_b is not None:
    ref = phase_b.loc[phase_b["model"] == "core (reference)", "f1_macro"].iloc[0]
    sd = phase_b.loc[phase_b["model"] == "core (reference)", "f1_macro_std"].iloc[0]
    d = phase_b.assign(delta=lambda x: x["f1_macro"] - ref)
    d = d[["model", "f1_macro", "delta"]].sort_values("delta", ascending=False).round(4)
    print(f"Reference: core, macro-F1 {ref:.4f} (fold SD {sd:.4f})\n")
    print(d.to_string(index=False))
    print(f"\nChanges smaller than the fold SD ({sd:.4f}) are noise, not improvements.")

Reference: core, macro-F1 0.2541 (fold SD 0.0019)

                      model  f1_macro   delta
core+context (506 features)    0.2605  0.0065
            rare pooling 1%    0.2547  0.0006
           core (reference)    0.2541  0.0000
         rare pooling 0.25%    0.2528 -0.0012
         want lists removed    0.2451 -0.0089
     SelectKBest chi2 k=300    0.2353 -0.0188
     SelectKBest chi2 k=200    0.2315 -0.0225
     SelectKBest chi2 k=all    0.2290 -0.0251
     SelectKBest chi2 k=100    0.2277 -0.0264

Changes smaller than the fold SD (0.0019) are noise, not improvements.


### How to read Phase B

The honest reading is that a difference smaller than the cross-validation standard
deviation is not a difference. Where a variant ties with the reference, the Stage 4
decision stands — and that is a result worth reporting, because it says the preprocessing
was already at a sensible point rather than that the experiment failed.

`core+context` is a special case: even if it scores higher, it is **not** adopted, because
the app's main users are students who cannot answer OrgSize or Industry. A model that
needs inputs the user does not have is worse in production than a slightly weaker model
that works. This is the train/serve mismatch the Stage 4 decision log flagged as risk 21.

## Phase C · Hyper-parameter search

**Randomised search, not grid search.** A grid over five hyper-parameters with four values
each is 1,024 fits per model; at 6–25 seconds a fit that is days of compute. Random search
over the same ranges finds a near-best setting in a few dozen fits, because typically only
two or three parameters matter and random sampling tries many distinct values of each
instead of a few repeated ones.

The search optimises **macro-F1**, the primary metric, with the same `StratifiedKFold`.
Boosting is searched over 3 folds instead of 5 purely for cost; the winning setting is
then re-scored over the standard 5 folds so every number in the final table is comparable.

In [5]:
for model, params in best_params.items():
    print(f"{model}\n    {params}")

Logistic Regression
    {'clf__C': 0.21751953118777648, 'clf__class_weight': None}
Linear SVM (calibrated)
    {'clf__estimator__C': 0.003550621427070771, 'clf__estimator__class_weight': 'balanced'}
Random Forest
    {'clf__n_estimators': 300, 'clf__min_samples_leaf': 4, 'clf__max_features': 0.1, 'clf__max_depth': 40, 'clf__class_weight': 'balanced'}
Hist Gradient Boosting
    {'clf__min_samples_leaf': 50, 'clf__max_leaf_nodes': 15, 'clf__max_iter': 200, 'clf__learning_rate': 0.2, 'clf__l2_regularization': 1.0}


In [6]:
phase_c[["model", "f1_macro", "f1_macro_std", "balanced_accuracy", "top3_accuracy",
         "log_loss", "family_f1_macro", "family_top3_accuracy", "fit_seconds"]].round(4)

,model,f1_macro,f1_macro_std,balanced_accuracy,top3_accuracy,log_loss,family_f1_macro,family_top3_accuracy,fit_seconds
0,Logistic Regression (tuned),0.3030,0.0082,0.2826,0.8387,1.4146,0.4294,0.8733,3.3470
1,Hist Gradient Boosting (tuned),0.2860,0.0096,0.2605,0.8309,1.4226,0.4030,0.8659,12.9254
2,Random Forest (tuned),0.2820,0.0081,0.2936,0.8131,1.7325,0.3585,0.8211,4.2030
3,Linear SVM (calibrated) (tuned),0.2536,0.0090,0.2300,0.8401,1.3998,0.3842,0.8749,4.2883


## Phase D · Tuned vs baseline, and the final decision

In [7]:
phase_d.round(4)

,model,baseline_f1_macro,tuned_f1_macro,gain,tuned_f1_macro_std,baseline_top3,tuned_top3,tuned_balanced_accuracy,tuned_log_loss,tuned_family_f1_macro,tuned_family_top3,fit_seconds
0,Logistic Regression,0.2547,0.3030,0.0483,0.0082,0.7262,0.8387,0.2826,1.4146,0.4294,0.8733,3.3470
1,Hist Gradient Boosting,0.2724,0.2860,0.0135,0.0096,0.8289,0.8309,0.2605,1.4226,0.4030,0.8659,12.9254
2,Random Forest,0.2537,0.2820,0.0283,0.0081,0.8301,0.8131,0.2936,1.7325,0.3585,0.8211,4.2030
3,Linear SVM (calibrated),0.1911,0.2536,0.0625,0.0090,0.8092,0.8401,0.2300,1.3998,0.3842,0.8749,4.2883


![tuned vs baseline](../reports/figures/05_tuned_vs_baseline.png)

In [8]:
print("Selection rule:\n ", decision["selection_rule"], "\n")
print("Best by macro-F1 :", decision["best_by_f1_macro"])
print("Tied within 1 SD :", ", ".join(decision["tied_within_1sd"]))
print("FINAL MODEL      :", decision["final_model"])
print("\nCross-validated scores of the final model:")
for k, v in decision["final_cv_scores"].items():
    print(f"  {k:<28}{v:>8.4f}")

Selection rule:
  highest 5-fold macro-F1 on the training split; models within one standard deviation of the best are treated as tied and the cheapest to fit wins, because the web backend reloads this pipeline and an unreliable improvement does not justify a slower model 

Best by macro-F1 : Logistic Regression
Tied within 1 SD : Logistic Regression
FINAL MODEL      : Logistic Regression

Cross-validated scores of the final model:
  f1_macro                      0.3030
  top3_accuracy                 0.8387
  balanced_accuracy             0.2826
  log_loss                      1.4146
  family_f1_macro               0.4294
  family_top3_accuracy          0.8733


### Why the final model is the final model

The selection rule is written down before the numbers are read, so it cannot be bent to
fit a favourite:

> Highest 5-fold macro-F1 on the training split. Models **within one standard deviation**
> of the best are treated as tied, and the cheapest to fit wins.

The tie-break exists because of what Stage 6 found: the gap between the strong models is
about the size of the fold-to-fold noise. Claiming the top model is "better" when it is
inside the noise band would be reading randomness. When two models are statistically
indistinguishable, the one that retrains in seconds and loads quickly in the backend is
the better engineering choice — and that reasoning, not a third decimal place, is what
justifies the pick.

Tuning gains in this project are small for every model. That is consistent with the Stage
6 finding that the ceiling is set by **overlapping roles**, not by hyper-parameters: no
setting of `C` separates Cloud Infrastructure from DevOps engineers when their technology
profiles have a cosine similarity of 0.99.

## The one test evaluation

`scripts/finalise_model.py` refits the chosen configuration on all 18,457 training rows
and scores it on the 4,615 held-out respondents — once. Running it repeatedly while
changing the model would turn the test split into a second validation set and the number
would stop meaning anything.

In [9]:
tp = C.REPORTS_DIR / "stage7_test_results.json"
if tp.exists():
    t = json.loads(tp.read_text())
    rows = [{"metric": k, "cross-validated": t["cv_scores"].get(k), "held-out test": v}
            for k, v in t["test_scores"].items()]
    display(pd.DataFrame(rows).round(4))
    print(f"\nmacro-F1 gap CV -> test: {t['cv_to_test_gap_f1_macro']:+.4f}")
    print(f"API parity: {t['api_parity']['rows_checked']} profiles through the web format, "
          f"max probability difference {t['api_parity']['max_probability_difference']:.2e}")
else:
    print("Run: python scripts/finalise_model.py")

,metric,cross-validated,held-out test
0,f1_macro,0.3030,0.2845
1,f1_weighted,NaN,0.5238
2,balanced_accuracy,0.2826,0.2657
3,accuracy,NaN,0.5493
4,top3_accuracy,0.8387,0.8392
5,log_loss,1.4146,1.4146
6,family_f1_macro,0.4294,0.4149
7,family_balanced_accuracy,NaN,0.3923
8,family_accuracy,NaN,0.5757
9,family_top3_accuracy,0.8733,0.8778



macro-F1 gap CV -> test: -0.0185
API parity: 300 profiles through the web format, max probability difference 0.00e+00


![test confusion](../reports/figures/05_test_confusion.png)

A small gap between the cross-validated estimate and the test score is the thing to look
for. It means the validation procedure was honest — no leak quietly inflated the CV
numbers — and that the model generalises to respondents it has never seen.

The **API parity check** is the bridge to Stages 9–10: it takes real training rows,
converts them into the JSON profile the web form will send, converts them back through
`profile_to_frame()`, and confirms the predictions are identical to the ones made
directly. That is what proves the deployed backend preprocesses exactly like the model
that was validated here.

Outputs for the next stages:

* `artifacts/model.joblib` — the fitted Pipeline the backend loads
* `artifacts/model_card.json` — what it is, how it scores, and what it must not be used for
* `reports/model_experiments.csv` — every run in the project, timestamped